# ARC-AGI-2 Backward Edit-Graph Induction v4

v3 failed with **0/50 demo coverage**, and its supposed anti-unification was actually exact intersection of already-instantiated schemas.

v4 tests the intended idea in stages:

1. **Object correspondence:** find shape-compatible input/output object mappings.
2. **Single-pair explanation:** derive concrete edit graphs that exactly transform each known input into its known output.
3. **Structural anti-unification:** replace concrete object identities with shared task-relative roles (`unique_size`, `left`, `unique_holey`, etc.) and shared relational destinations/reference colors.
4. **Train replay:** a generalized schema counts only if it reconstructs **every** training output exactly.
5. **Test execution:** only then is the schema applied to unseen test inputs.

### Frozen diagnostic gates for the first run

Run exactly **20 development tasks** first.

- Continue only if at least **30% of individual training pairs** receive a valid concrete edit explanation.
- Continue the anti-unification branch only if at least **20% of tasks** obtain a generalized schema that reconstructs every demonstration.

This is a CPU-first symbolic experiment. **GPU is not needed.**


In [ ]:
LIMIT = 20
SPLIT = "dev"

print("LIMIT =", LIMIT)
print("SPLIT =", SPLIT)
print("CPU-only experiment; GPU is not required.")


In [ ]:
from pathlib import Path
import importlib.util, requests, sys

REPO = "https://raw.githubusercontent.com/Vedsaga/arc-agi/main"
WORK = Path("/kaggle/working")
module_path = WORK / "edit_graph_induction_v4.py"

url = f"{REPO}/research/experiments/edit_graph_induction_v4.py"
r = requests.get(url, timeout=30)
r.raise_for_status()
module_path.write_bytes(r.content)
print("Downloaded", url)

spec = importlib.util.spec_from_file_location("edit_graph_induction_v4", module_path)
v4 = importlib.util.module_from_spec(spec)
sys.modules["edit_graph_induction_v4"] = v4
spec.loader.exec_module(v4)
print("Loaded v4")


In [ ]:
tasks, pairs, summary = v4.run(limit=LIMIT, split=SPLIT)
summary


## Stage diagnostics

The point of v4 is to identify **where** the idea succeeds or fails.

- If `pair_explanation_rate < 0.30`: stop extending this handcrafted symbolic branch.
- If pair explanations pass but `task_train_reconstruction_rate < 0.20`: the concrete representation has some coverage, but the anti-unifier/role language is the bottleneck.
- If both gates pass: proceed to a controlled forward-search-vs-backward-induction ablation.
- Test accuracy is reported, but it is **not** used to rescue a failed representation stage.


In [ ]:
print("Decision:", summary["decision"])
print("Pair explanation rate:", summary["pair_explanation_rate"])
print("Task train-reconstruction rate:", summary["task_train_reconstruction_rate"])
print("Pass@2:", summary["pass2_tasks"], "/", summary["tasks"])

display(tasks[[
    "task",
    "train_examples",
    "pair_explainable_count",
    "mean_edit_explanations",
    "anti_unified_schema_count",
    "train_reconstruction",
    "test_exact_pass2",
    "seconds",
]])

print("\nTraining pairs with zero concrete explanation:")
display(pairs[~pairs.pair_explainable].head(30))


## Files to share

After the run, upload these three files from `/kaggle/working/edit_graph_v4/`:

- `edit_graph_v4_dev_summary.json`
- `edit_graph_v4_dev.csv`
- `edit_graph_v4_dev_pairs.csv`

Do **not** switch to the official ARC evaluation set yet.
